# 02 - Transcription Mistral des audios restants

Ce notebook utilise les audios deja presents dans `audio/` et ne traite que ceux qui n'ont pas encore une transcription Mistral reussie. Lancez les cellules dans l'ordre : vous entrez seulement la cle Mistral quand elle est demandee. La cle est masquee, jamais enregistree dans le notebook, puis supprimee de l'environnement a la fin.

In [ ]:
from pathlib import Path
import getpass
import os
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from benchmark import discover_existing_audio_samples, transcribe_samples

In [ ]:
# La cle saisie est masquee et sa valeur n'est pas sauvegardee dans la cellule.
if not os.getenv('MISTRAL_API_KEY'):
    os.environ['MISTRAL_API_KEY'] = getpass.getpass('Cle Mistral : ')

from mistralai.client import Mistral
with Mistral(api_key=os.environ['MISTRAL_API_KEY']) as client:
    model_response = client.models.list()

available_model_ids = {
    getattr(model, 'id', '') for model in (model_response.data or [])
}
preferred_models = [
    'darija-stt-solutions-2607',
    'voxtral-mini-2602',
    'voxtral-mini-latest',
]
selected_model = next(
    (model_id for model_id in preferred_models if model_id in available_model_ids),
    None,
)
if selected_model is None:
    raise RuntimeError(
        'Aucun modele de transcription compatible trouve pour cette cle.'
    )
os.environ['MISTRAL_MODEL'] = selected_model

print('Cle configuree :', bool(os.getenv('MISTRAL_API_KEY')))
print('Modele selectionne :', selected_model)

In [ ]:
# Selectionne tous les audios encore sans transcription Mistral reussie.
# LIMIT = 0 veut dire : traiter tous les audios en attente.
LIMIT = 0
selected = discover_existing_audio_samples(limit=LIMIT, pending_only=True)
print(f'{len(selected)} audio(s) restant(s) a envoyer a Mistral.')
[(row['video_id'], row['title']) for row in selected]

In [ ]:
# Verification lisible, sans afficher la cle.
print('Modele retenu :', os.environ['MISTRAL_MODEL'])
print('Audios a traiter :', len(selected))
sorted(
    model_id for model_id in available_model_ids
    if 'voxtral' in model_id.lower() or 'darija' in model_id.lower()
)

Apres le traitement, verifiez que le resume affiche `Successful transcriptions` egal au nombre d'audios restants. Ensuite lancez le nettoyage de la cle, puis passez au notebook `03_run_ctc_quality_check.ipynb`.

In [ ]:
if not selected:
    print('Aucun audio restant : toutes les transcriptions Mistral sont deja reussies.')
else:
    summary = transcribe_samples(selected, force=False)
    display(summary)

In [ ]:
# Nettoyage de la cle apres le traitement. Redemarrez aussi le kernel si vous voulez etre extra prudent.
os.environ.pop('MISTRAL_API_KEY', None)
os.environ.pop('MISTRAL_MODEL', None)
print("Cle supprimee de l'environnement du notebook.")